In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

c:\Users\harini jeyashree\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Here I'm going to read the csv and set date column as index

In [3]:
df = pd.read_csv('data.csv', parse_dates=['Date'], index_col='Date')
print(df.head())

            Temperature
Date                   
2010-01-01    27.483571
2010-01-02    24.308678
2010-01-03    28.238443
2010-01-04    32.615149
2010-01-05    23.829233


Now, using MinMaxScaler I scaled the datas between range(0,1)

In [4]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df.values)

Creating a sliding window function to create data sets for X & y lists where X is the question and y is the ans for model to learn. 

In [5]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), 0])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)


time_step = 100
X, y = create_dataset(scaled_data, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)

Then converted X & y as numpy arrays and reshaped it to train the model

After that, created a sequential model for this time series analysis with GRU layers and Adam optimizer to optimize model predictions with mean_squared_error


In [6]:
model = Sequential()
model.add(GRU(units=50, return_sequences=True, input_shape=(X.shape[1], 1)))
model.add(GRU(units=50))
model.add(Dense(units=1))
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

c:\Users\harini jeyashree\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Asking the model to go through data sets 10 times with 32 chunks of data at once to learn and train

In [7]:
model.fit(X, y, epochs=10, batch_size=32)

Epoch 1/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 28s 85ms/step - loss: 0.0234
Epoch 2/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 101ms/step - loss: 0.0180
Epoch 3/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 101ms/step - loss: 0.0179
Epoch 4/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 24s 96ms/step - loss: 0.0178
Epoch 5/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 100ms/step - loss: 0.0180
Epoch 6/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 27s 110ms/step - loss: 0.0178
Epoch 7/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 30s 123ms/step - loss: 0.0178
Epoch 8/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 27s 110ms/step - loss: 0.0178
Epoch 9/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 30s 122ms/step - loss: 0.0178
Epoch 10/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 100ms/step - loss: 0.0177


After successfull training the model is evaluated with new 1 window of 100 days data to predict 1 feature (temperature)

In [8]:
input_sequence = scaled_data[-time_step:].reshape(1, time_step, 1)
predicted_values = model.predict(input_sequence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 846ms/step


As the datas were scaled previously between 0 and 1 now to display the output I'm using inverse transform to reverse the data back to original

In [9]:
predicted_values = scaler.inverse_transform(predicted_values)
print(
    f"The predicted temperature for the next day is: {predicted_values[0][0]:.2f}°C")

The predicted temperature for the next day is: 24.99°C
